<a href="https://colab.research.google.com/github/eniompw/TinyLM/blob/main/TinyMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from datasets import load_dataset
import itertools
import cupy as cp

def load_tinystories(num_records=200, context_size=4):
    """
    Fetches the TinyStories dataset and prepares it for a character-level language model.

    Returns:
        inputs (cp.ndarray): Sliding window context arrays of shape (N, context_size)
        targets (cp.ndarray): Target next-character arrays of shape (N,)
        vocab (list): The list of unique characters (vocabulary)
        encoded (list): The full encoded text as a list of integer IDs
    """
    # Fetch data
    dataset = load_dataset('karpathy/tinystories-gpt4-clean', split='train', streaming=True)
    text = ''.join(s['text'] for s in itertools.islice(dataset, num_records))

    # Build vocabulary and tokenize
    vocab = sorted(set(text))                                       # ordered list of unique characters
    char_to_id = {c: i for i, c in enumerate(vocab)}                # dictionary mapping char to integer id
    encoded = [char_to_id[c] for c in text]                         # map entire text to integer sequence

    # Create sliding windows for inputs and targets
    inputs = cp.array([encoded[i:i+context_size] for i in range(len(encoded)-context_size)]) # sliding windows
    targets = cp.array(encoded[context_size:])                                                # next char to predict

    return inputs, targets, vocab, encoded

In [4]:
import warnings, time, cupy as cp, numpy as np
#from tinystories_dataset import load_tinystories

warnings.filterwarnings('ignore')

def softmax(logits):                                            # converts raw network outputs to probabilities
    e = cp.exp(logits - logits.max(axis=1, keepdims=True))      # shift to prevent float overflow
    return e / e.sum(axis=1, keepdims=True)                     # normalize so all probabilities sum to 1

# --- Data & Tokenization ---
context_size = 4                                                # number of previous chars used to predict next
inputs, targets, vocab, encoded = load_tinystories(num_records=200, context_size=context_size)

vocab_size = len(vocab)                                         # total count of unique characters
N = len(inputs)                                                 # total number of training examples

# --- Model ---
emb_dim, hidden_size = 256, 150                                 # embedding dimensions, hidden layer neurons
cp.random.seed(42); randn = lambda *s: (cp.random.randn(*s) * 0.1).astype(cp.float32) # seed & normal init helper
C  = randn(vocab_size, emb_dim)                                 # token embedding lookup matrix
W1 = randn(context_size * emb_dim, hidden_size)                 # weights mapping context to hidden state
W2 = randn(hidden_size, vocab_size)                             # weights mapping hidden state to logits

# --- Train ---
lr, batch_size = 0.5, 1024                                      # learning rate, number of samples per batch
start = time.time()                                             # track training duration
for epoch in range(2001):
    idx = cp.random.randint(0, N, size=batch_size)              # random array of indices for batch
    X, Y = inputs[idx], targets[idx]                            # fetch random mini-batch (inputs and labels)

    # Forward pass
    emb = C[X].reshape(batch_size, -1)                          # concatenate embeddings for the window
    h   = cp.maximum(0, emb @ W1)                               # apply ReLU non-linearity to hidden state
    probs = softmax(h @ W2)                                     # get probability distribution over vocab

    # Backward pass
    probs[cp.arange(batch_size), Y] -= 1                        # in-place CE gradient: (probs - 1) for true labels
    probs /= batch_size                                         # average loss over batch (probs is now dlogits)

    dW2, dh = h.T @ probs, (probs @ W2.T) * (h > 0)             # gradients for output weights & hidden state (ReLU backprop)
    dW1 = emb.T @ dh                                            # gradient for hidden weights

    dC = cp.zeros_like(C)                                       # gradient accumulator for embeddings
    cp.add.at(dC, X.ravel(), (dh @ W1.T).reshape(-1, emb_dim))  # safely accumulate grads for repeated tokens

    for p, g in zip((C, W1, W2), (dC, dW1, dW2)):
        p -= lr * g                                             # standard SGD parameter update

    if epoch % 200 == 0:
        logits = cp.maximum(0, C[inputs].reshape(N, -1) @ W1) @ W2  # full dataset forward pass
        preds = logits.argmax(1)                                    # mathematical shortcut: argmax directly on logits
        print(f"Epoch {epoch:4d} | Acc: {cp.mean(preds == targets):.1%}")

print(f"Training time: {time.time() - start:.1f}s")

# --- Generate ---
def generate(num_chars=200):
    ctx = list(encoded[:context_size])                          # start with true initial context from text
    out = [vocab[i] for i in ctx]                               # decode initial context to string
    for _ in range(num_chars):
        p = cp.asnumpy(softmax(cp.maximum(0, C[cp.array([ctx])].reshape(1, -1) @ W1) @ W2)[0]) # fused forward pass
        next_id = int(np.random.choice(vocab_size, p=p))        # randomly sample from predicted distribution
        ctx = ctx[1:] + [next_id]                               # slide context window forward by one token
        out.append(vocab[next_id])
    return ''.join(out)

print(generate())

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

Epoch    0 | Acc: 4.7%
Epoch  200 | Acc: 44.8%
Epoch  400 | Acc: 49.0%
Epoch  600 | Acc: 52.3%
Epoch  800 | Acc: 55.0%
Epoch 1000 | Acc: 56.4%
Epoch 1200 | Acc: 56.7%
Epoch 1400 | Acc: 58.2%
Epoch 1600 | Acc: 58.3%
Epoch 1800 | Acc: 59.2%
Epoch 2000 | Acc: 59.3%
Training time: 8.3s
Once upon a toy came nex, then, the poincher they mage. You mowent tayt dogen. She was so hax lach in mom andig.
The moge!"
The bill. She oden. Tom and was a pill kick
tadbyaning eaming hox, the happletel
